# StreetFoodAI: Google Colab GPU Training & Evaluation Pipeline

This notebook trains and evaluates **MobileNetV3-Large** and **EfficientNet-B0** on Google Colab GPU hardware (T4 / A100), executes Grad-CAM visual attention mapping, and exports models to ONNX.

In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Clone Repository or Install Dependencies
# If running inside cloned workspace:
!pip install -r backend/requirements.txt

In [ ]:
# 3. Run Dataset Preparation and Stratified Splitting
from ml.data.prep_dataset import prepare_and_split_dataset

report = prepare_and_split_dataset(
    raw_dir="data/raw",
    processed_dir="data/processed",
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    random_seed=42
)
print(f"Valid images: {report['valid_images']}, Duplicates removed: {len(report['duplicate_images'])}")

In [ ]:
# 4. Fine-Tune MobileNetV3-Large
from ml.training.train import run_training

mobilenet_results = run_training(
    model_name="mobilenet_v3_large",
    data_dir="data/processed",
    epochs=25,
    batch_size=32,
    lr=1e-4,
    patience=5,
    use_mlflow=False
)

In [ ]:
# 5. Fine-Tune EfficientNet-B0
efficientnet_results = run_training(
    model_name="efficientnet_b0",
    data_dir="data/processed",
    epochs=25,
    batch_size=32,
    lr=1e-4,
    patience=5,
    use_mlflow=False
)

In [ ]:
# 6. Evaluate Models on Held-Out Test Set
from ml.evaluation.evaluate import evaluate_model
from ml.evaluation.compare import generate_model_comparison

evaluate_model(model_name="mobilenet_v3_large", weights_path="ml/checkpoints/best_mobilenet_v3_large.pth")
evaluate_model(model_name="efficientnet_b0", weights_path="ml/checkpoints/best_efficientnet_b0.pth")
comparison = generate_model_comparison()

In [ ]:
# 7. Generate Grad-CAM Attention Heatmap for a Sample Dish
from PIL import Image
from ml.inference.predictor import StreetFoodPredictor
from ml.explainability.gradcam import GradCAM, overlay_heatmap
from ml.data.augmentation import get_raw_display_transforms
import matplotlib.pyplot as plt

predictor = StreetFoodPredictor(model_name="mobilenet_v3_large", weights_path="ml/checkpoints/best_mobilenet_v3_large.pth")
test_img = Image.open("data/raw/sample_test/vada_pav.jpg").convert("RGB")

model, target_layer = predictor.get_model_for_gradcam()
gradcam = GradCAM(model=model, target_layer=target_layer)
tensor = predictor.transform(test_img).unsqueeze(0).to(predictor.device)
cam, cls_idx, conf = gradcam.generate_heatmap(tensor)
gradcam.remove_hooks()

overlay = overlay_heatmap(get_raw_display_transforms()(test_img), cam)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(test_img)
axes[0].set_title("Original Input")
axes[1].imshow(overlay)
axes[1].set_title(f"Grad-CAM Heatmap ({predictor.classes[cls_idx]}: {conf*100:.1f}%)")
plt.show()

In [ ]:
# 8. Export to ONNX
from ml.inference.onnx_export import export_to_onnx
export_to_onnx(
    model_name="mobilenet_v3_large",
    weights_path="ml/checkpoints/best_mobilenet_v3_large.pth",
    output_path="ml/checkpoints/mobilenet_v3_large.onnx"
)